# Chapter 10: Verdicts, Failures, and Severity Classification

**Learning objectives:**
- Aggregate sub-verdicts into a TypedVerdict with full audit trail
- Understand the failure code taxonomy (20+ codes across 7 categories)
- Classify failure severity with risk-tier awareness (CRITICAL / HIGH / MODERATE / LOW)
- Interpret the SeverityAssessment composite: impact x recoverability x detectability

---
## Preamble

Chapter 9 showed how individual verifiers produce `SubVerdict` objects — one per claim. But a single evaluation question may yield multiple claims and therefore multiple SubVerdicts. Stakeholders need an **aggregate verdict** for the question as a whole, along with a **prioritized list of failures** classified by severity.

This chapter introduces three components that close Part III:

1. **TypedVerdict** — aggregates per-claim sub-verdicts into an overall verdict with backwards-compatible geometric scores and a full audit trail.
2. **FailureCode taxonomy** — a controlled vocabulary of 20+ failure codes organized into seven categories, each mapping to a specific GMS verification route.
3. **SeverityClassifier** — maps each failure to CRITICAL / HIGH / MODERATE / LOW using three inputs: failure code, risk tier and GMS geometric signals.

Key modules:
- `harness.doe.verdict` — `TypedVerdict`
- `harness.doe.failures` — `FailureCode`, `FailureAttribution`, `FailureSeverity`
- `harness.doe.severity` — `SeverityLevel`, `SeverityAssessment`, `SeverityClassifier`

In [ ]:
# === Preamble (all notebooks) ===
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
# [AUTO-INJECTED] Disable DOE tracing for parallel execution safety
from knowlytix.harness.doe import DOEHarnessConfig
DOEHarnessConfig.__dataclass_fields__['enable_tracing'].default = False


In [ ]:
from importlib.resources import files
# Load the Chapter 4 store and set up the benchmark
from knowlytix.harness.doe import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.doe").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)

bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"\nStore: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## The FailureCode Taxonomy

Every verification failure carries a `FailureCode` from a controlled vocabulary. The codes are organized into seven categories that map to GMS verification routes:

In [ ]:
# Import the full failure taxonomy
from knowlytix.harness.doe.failures import FailureCode, FailureAttribution

# Organize codes by category
categories = {
    "Semantic": ["semantic_extraction_failure", "semantic_mismatch",
                 "hallucination_detected", "incomplete_answer",
                 "no_claims_extracted"],
    "Graph/Structural": ["entity_not_found", "relation_not_found",
                         "fact_incorrect", "path_inconsistency"],
    "Logic": ["contradiction_detected", "tension_exceeded",
              "holonomy_defect"],
    "Numeric": ["threshold_violation", "boundary_failure"],
    "Artifact/ENM": ["exact_mismatch", "artifact_integrity_failure"],
    "Policy/Agentic": ["tool_call_invalid", "policy_violation",
                        "policy_blocked_pre_execution", "approval_required"],
    "Process": ["required_tool_missing", "forbidden_tool_called",
                "process_deviation", "tool_budget_exceeded"],
}

total_codes = sum(len(v) for v in categories.values())
print(f"FailureCode taxonomy: {total_codes} codes in {len(categories)} categories\n")

print(f"{'Category':<20s} {'Codes':>5s}  {'Example'}")
print("-" * 65)
for cat, codes in categories.items():
    print(f"{cat:<20s} {len(codes):>5d}  {codes[0]}")

In [5]:
# Full listing of all failure codes
print(f"{'Code':<35s} {'GMS Method'}")
print("-" * 60)

code_to_method = {
    "entity_not_found": "fuzzy_match_entity",
    "relation_not_found": "query_triples",
    "fact_incorrect": "score_triple",
    "path_inconsistency": "check_holonomy",
    "contradiction_detected": "tension_energy",
    "tension_exceeded": "tension_energy",
    "holonomy_defect": "check_holonomy",
    "threshold_violation": "check_threshold",
    "boundary_failure": "check_threshold",
    "exact_mismatch": "enm.read_exact + SHA-256",
    "artifact_integrity_failure": "enm.verify_integrity",
    "tool_call_invalid": "score_triple (tool)",
    "policy_violation": "fuzzy_match_entity",
    "hallucination_detected": "LLM judge + GMS",
    "semantic_mismatch": "LLM judge + GMS",
    "verification_error": "(infrastructure)",
}

for code in FailureCode:
    method = code_to_method.get(code.value, "—")
    print(f"{code.value:<35s} {method}")

Code                                GMS Method
------------------------------------------------------------
semantic_extraction_failure         —
semantic_mismatch                   LLM judge + GMS
hallucination_detected              LLM judge + GMS
incomplete_answer                   —
no_claims_extracted                 —
entity_not_found                    fuzzy_match_entity
relation_not_found                  query_triples
fact_incorrect                      score_triple
path_inconsistency                  check_holonomy
contradiction_detected              tension_energy
tension_exceeded                    tension_energy
holonomy_defect                     check_holonomy
threshold_violation                 check_threshold
boundary_failure                    check_threshold
exact_mismatch                      enm.read_exact + SHA-256
artifact_integrity_failure          enm.verify_integrity
tool_call_invalid                   score_triple (tool)
policy_violation                    fu

---

## The TypedVerdict

A `TypedVerdict` aggregates per-claim sub-verdicts into an overall verdict. It carries:
- **Backwards-compatible geometric fields**: `geodesic_confidence`, `tension_score`, `holonomy_score`, `overall_confidence`
- **Audit trail**: `claims`, `sub_verdicts`, `failures`
- **Counts**: `n_claims`, `n_passed`, `n_failed`
- **Provenance**: `run_id`, `question_id`, `timestamp`

The `from_sub_verdicts()` factory method computes the overall confidence using the weighted formula:

$$\text{overall} = 0.3 \times \text{exact} + 0.3 \times e^{-\bar{d}} + 0.2 \times \left(1 - \frac{\bar{\tau}}{2}\right) + 0.2 \times e^{-\bar{h}}$$

where $\bar{d}$ is the mean geodesic, $\bar{\tau}$ is the mean tension and $\bar{h}$ is the mean holonomy defect.

In [ ]:
# Import verifiers and verdict
from knowlytix.harness.doe.verifiers import (
    VerificationRouter,
)
from knowlytix.harness.doe.claims import (
    ClaimProvenance, FactClaim, ClaimCompiler,
)
from knowlytix.harness.doe.verdict import TypedVerdict

print("TypedVerdict fields:")
for f_name in ["correct", "geodesic_confidence", "tension_score",
               "holonomy_score", "overall_confidence",
               "n_claims", "n_passed", "n_failed",
               "run_id", "question_id", "timestamp"]:
    print(f"  {f_name}")

---

## Lab 10.1 — Building a Complete TypedVerdict

We construct claims, verify them individually and aggregate the results into a `TypedVerdict`.

In [7]:
# Lab 10.1a: Construct claims and verify them
router = VerificationRouter(store)

# Three claims of different types
claims = [
    FactClaim(
        head="MDL-001", relation="has_risk_rating", tail="High",
        raw_text="MDL-001 has risk rating High",
        confidence=1.0,
        provenance=ClaimProvenance(extraction_method="metadata"),
    ),
    FactClaim(
        head="MDL-001", relation="has_downstream_model", tail="MDL-008",
        raw_text="MDL-001 feeds into MDL-008",
        confidence=1.0,
        provenance=ClaimProvenance(extraction_method="metadata"),
    ),
    FactClaim(
        head="MDL-001", relation="has_risk_rating", tail="Low",
        raw_text="MDL-001 has risk rating Low",
        confidence=1.0,
        provenance=ClaimProvenance(extraction_method="text_match"),
    ),
]

# Verify each claim
sub_verdicts = router.route_all(claims)

print(f"{'#':<3s} {'Type':<10s} {'Passed':>7s} {'Conf':>6s} {'GMS Method':<20s} {'Detail'}")
print("-" * 80)
for i, (c, sv) in enumerate(zip(claims, sub_verdicts), 1):
    detail = sv.detail[:40] + "..." if len(sv.detail) > 40 else sv.detail
    print(f"{i:<3d} {c.claim_type:<10s} {str(sv.passed):>7s} "
          f"{sv.confidence:>6.3f} {sv.gms_method_used:<20s} {detail}")

#   Type        Passed   Conf GMS Method           Detail
--------------------------------------------------------------------------------
1   fact         False  0.219 score_triple         (MDL-001, has_risk_rating, High) geodesi...
2   fact          True  0.324 score_triple         (MDL-001, has_downstream_model, MDL-008)...
3   fact         False  0.260 score_triple         (MDL-001, has_risk_rating, Low) geodesic...


In [8]:
# Lab 10.1b: Aggregate into a TypedVerdict
verdict = TypedVerdict.from_sub_verdicts(
    claims=claims,
    sub_verdicts=sub_verdicts,
    exact_correct=True,  # traditional scorer says correct
    exact_detail="Exact match: High",
    run_id="lab10-demo",
    question_id="q-demo-001",
)

print("=== TypedVerdict ===")
print(f"  correct:              {verdict.correct}")
print(f"  geodesic_confidence:  {verdict.geodesic_confidence:.4f}")
print(f"  tension_score:        {verdict.tension_score:.4f}")
print(f"  holonomy_score:       {verdict.holonomy_score:.4f}")
print(f"  overall_confidence:   {verdict.overall_confidence:.4f}")
print(f"  n_claims:             {verdict.n_claims}")
print(f"  n_passed:             {verdict.n_passed}")
print(f"  n_failed:             {verdict.n_failed}")
print(f"  failures:             {len(verdict.failures)}")
print(f"  failure_codes:        {verdict.failure_codes}")
print(f"  run_id:               {verdict.run_id}")
print(f"  timestamp:            {verdict.timestamp}")

=== TypedVerdict ===
  correct:              True
  geodesic_confidence:  0.2640
  tension_score:        0.0000
  holonomy_score:       0.0000
  overall_confidence:   0.7792
  n_claims:             3
  n_passed:             1
  n_failed:             2
  failures:             2
  failure_codes:        ['fact_incorrect']
  run_id:               lab10-demo
  timestamp:            2026-04-15 22:12:29


In [9]:
# Lab 10.1c: Verifier summary
print("\nVerifier summary:")
for vname, counts in verdict.verifier_summary.items():
    print(f"  {vname}: passed={counts['passed']}, "
          f"failed={counts['failed']}, total={counts['total']}")

# Inspect the failure detail
print("\nFailure details:")
for f in verdict.failures:
    print(f"  [{f.code.value}] verifier={f.verifier}")
    print(f"    gms_method: {f.gms_method}")
    print(f"    raw_score:  {f.raw_score}")
    print(f"    detail:     {f.detail[:80]}")


Verifier summary:
  GraphVerifier: passed=1, failed=2, total=3

Failure details:
  [fact_incorrect] verifier=GraphVerifier
    gms_method: score_triple
    raw_score:  1.5189634561538696
    detail:     Triple (MDL-001, has_risk_rating, High) geodesic=1.5190 > threshold=1.2
  [fact_incorrect] verifier=GraphVerifier
    gms_method: score_triple
    raw_score:  1.3487706184387207
    detail:     Triple (MDL-001, has_risk_rating, Low) geodesic=1.3488 > threshold=1.2


The TypedVerdict aggregates the three sub-verdicts into backwards-compatible geometric scores. The `geodesic_confidence` is computed as $e^{-\bar{d}}$ where $\bar{d}$ is the mean geodesic across all GraphVerifier sub-verdicts. The `overall_confidence` combines exact match, geodesic, tension and holonomy using the weighted formula.

---

## Severity Classification

Not all failures are equally serious. A `fact_incorrect` in a credit decisioning agent is more dangerous than a `relation_not_found` in an advisory chatbot. The `SeverityClassifier` uses three inputs to classify each failure:

1. **Failure code** — static base severity from `CODE_SEVERITY_MAP`
2. **Risk tier** — `action_taking` escalates moderate failures to high
3. **GMS geometric signals** — raw geodesic/tension/holonomy scores modulate impact and detectability

The four severity levels are:

| Level | Meaning |
|---|---|
| CRITICAL | Safety-relevant, unrecoverable, hard to detect |
| HIGH | Functionally wrong, potentially recoverable |
| MODERATE | Partially wrong, recoverable |
| LOW | Minor, cosmetic, or informational |

In [ ]:
# Import severity classification
from knowlytix.harness.doe.severity import (
    SeverityClassifier, SeverityLevel, CODE_SEVERITY_MAP,
)

# Show the default code-to-severity mapping
print(f"Default CODE_SEVERITY_MAP ({len(CODE_SEVERITY_MAP)} codes):\n")

by_level = {}
for code, level in CODE_SEVERITY_MAP.items():
    by_level.setdefault(level.value, []).append(code.value)

for level_name in ["critical", "high", "moderate", "low"]:
    codes = by_level.get(level_name, [])
    print(f"  {level_name.upper()} ({len(codes)} codes):")
    for c in sorted(codes):
        print(f"    - {c}")
    print()

In [11]:
# Lab 10.1d: Classify failure severity
classifier = SeverityClassifier(store=store)

# Classify the failures from our verdict
print("Severity classification (standard tier):\n")
for f in verdict.failures:
    assessment = classifier.classify(f, risk_tier="standard")
    print(f"  [{f.code.value}]")
    print(f"    Level:          {assessment.level.value}")
    print(f"    Impact:         {assessment.impact_score:.3f}")
    print(f"    Recoverability: {assessment.recoverability_score:.3f}")
    print(f"    Detectability:  {assessment.detectability_score:.3f}")
    print(f"    Composite:      {assessment.composite_score:.3f}")
    print(f"    Rationale:      {assessment.rationale}")
    print()

Severity classification (standard tier):

  [fact_incorrect]
    Level:          high
    Impact:         0.967
    Recoverability: 0.333
    Detectability:  0.600
    Composite:      0.750
    Rationale:      base=high (code=fact_incorrect); gms_score_triple=1.5190

  [fact_incorrect]
    Level:          high
    Impact:         0.967
    Recoverability: 0.333
    Detectability:  0.600
    Composite:      0.750
    Rationale:      base=high (code=fact_incorrect); gms_score_triple=1.3488



In [12]:
# Compare severity across risk tiers
tiers = ["standard", "advisory", "recommendation", "action_taking"]

print("Risk-tier escalation for fact_incorrect (base=HIGH):\n")
print(f"{'Tier':<18s} {'Level':<12s} {'Impact':>7s} {'Recover':>8s} "
      f"{'Detect':>7s} {'Composite':>10s}")
print("-" * 68)

test_failure = FailureAttribution(
    code=FailureCode.FACT_INCORRECT,
    claim_id="test",
    verifier="GraphVerifier",
    gms_method="score_triple",
    raw_score=1.47,
)

for tier in tiers:
    a = classifier.classify(test_failure, risk_tier=tier)
    print(f"{tier:<18s} {a.level.value:<12s} {a.impact_score:>7.3f} "
          f"{a.recoverability_score:>8.3f} {a.detectability_score:>7.3f} "
          f"{a.composite_score:>10.3f}")

Risk-tier escalation for fact_incorrect (base=HIGH):

Tier               Level         Impact  Recover  Detect  Composite
--------------------------------------------------------------------
standard           high           0.967    0.333   0.600      0.750
advisory           high           0.967    0.333   0.600      0.750
recommendation     high           1.000    0.333   0.600      0.767
action_taking      critical       1.000    0.333   0.600      0.767


In [13]:
# Compare escalation across different failure codes
test_codes = [
    (FailureCode.HALLUCINATION_DETECTED, "score_triple", 0.9),
    (FailureCode.FACT_INCORRECT, "score_triple", 1.47),
    (FailureCode.HOLONOMY_DEFECT, "check_holonomy", 0.3),
    (FailureCode.EXACT_MISMATCH, "enm.verify_integrity", 0.001),
    (FailureCode.POLICY_VIOLATION, "fuzzy_match_entity", None),
    (FailureCode.VERIFICATION_ERROR, "routing", None),
]

print("\nSeverity comparison: standard vs action_taking tier:\n")
print(f"{'Code':<30s} {'Standard':<12s} {'Action-Taking':<14s} {'Escalated?'}")
print("-" * 65)

for code, method, score in test_codes:
    f = FailureAttribution(
        code=code, claim_id="t", verifier="test",
        gms_method=method, raw_score=score,
    )
    a_std = classifier.classify(f, risk_tier="standard")
    a_act = classifier.classify(f, risk_tier="action_taking")
    escalated = "YES" if a_std.level != a_act.level else "no"
    print(f"{code.value:<30s} {a_std.level.value:<12s} "
          f"{a_act.level.value:<14s} {escalated}")


Severity comparison: standard vs action_taking tier:

Code                           Standard     Action-Taking  Escalated?
-----------------------------------------------------------------
hallucination_detected         critical     critical       no
fact_incorrect                 high         critical       YES
holonomy_defect                moderate     high           YES
exact_mismatch                 low          low            no
policy_violation               critical     critical       no
verification_error             low          low            no


The `action_taking` tier escalates MODERATE and HIGH failures by one level:
- `fact_incorrect` (HIGH) -> CRITICAL in action_taking
- `holonomy_defect` (MODERATE) -> HIGH in action_taking

This means an action-taking agent (e.g., one that executes trades or approves loans) faces stricter scrutiny than an advisory agent.

---

## The SeverityAssessment Composite

Each `SeverityAssessment` carries three dimension scores:

- **Impact** ($[0,1]$): How much damage the failure could cause. Influenced by code severity, geodesic distance and risk tier.
- **Recoverability** ($[0,1]$): How easy it is to recover. Low-severity codes and low tension are more recoverable.
- **Detectability** ($[0,1]$): How easy it is to detect externally. Structural failures (exact mismatch) are highly detectable; subtle hallucinations near plausible entities are not.

The composite score combines them:
$$\text{composite} = 0.50 \times \text{impact} + 0.25 \times (1 - \text{recoverability}) + 0.25 \times (1 - \text{detectability})$$

If composite $\ge 0.85$, the severity is escalated one additional level (geometric modulation).

In [14]:
# Demonstrate the three dimensions on diverse failure types
diverse_failures = [
    ("Hallucination (close)", FailureCode.HALLUCINATION_DETECTED,
     "score_triple", 0.5),
    ("Hallucination (far)", FailureCode.HALLUCINATION_DETECTED,
     "score_triple", 1.8),
    ("Fact incorrect", FailureCode.FACT_INCORRECT,
     "score_triple", 1.47),
    ("Holonomy defect", FailureCode.HOLONOMY_DEFECT,
     "check_holonomy", 0.3),
    ("Exact mismatch", FailureCode.EXACT_MISMATCH,
     "enm.verify_integrity", 0.001),
    ("Tension exceeded", FailureCode.TENSION_EXCEEDED,
     "tension_energy", 1.5),
    ("Policy violation", FailureCode.POLICY_VIOLATION,
     "fuzzy_match_entity", None),
]

print(f"{'Failure':<25s} {'Level':<10s} {'Impact':>7s} {'Recover':>8s} "
      f"{'Detect':>7s} {'Compos':>7s}")
print("-" * 70)

for label, code, method, score in diverse_failures:
    f = FailureAttribution(
        code=code, claim_id="t", verifier="test",
        gms_method=method, raw_score=score,
    )
    a = classifier.classify(f, risk_tier="standard")
    print(f"{label:<25s} {a.level.value:<10s} {a.impact_score:>7.3f} "
          f"{a.recoverability_score:>8.3f} {a.detectability_score:>7.3f} "
          f"{a.composite_score:>7.3f}")

Failure                   Level       Impact  Recover  Detect  Compos
----------------------------------------------------------------------
Hallucination (close)     critical     1.000    0.000   0.250   0.938
Hallucination (far)       critical     1.000    0.000   0.800   0.800
Fact incorrect            high         0.967    0.333   0.600   0.750
Holonomy defect           moderate     0.333    0.667   0.700   0.325
Exact mismatch            low          0.000    1.000   0.950   0.013
Tension exceeded          moderate     0.333    0.267   0.700   0.425
Policy violation          critical     1.000    0.000   0.500   0.875


---

## Full Pipeline: DOE Questions to Severity-Classified Verdicts

Let us run the complete Part III pipeline on all 640 DOE questions:
1. Generate DOE questions
2. Compile claims (ClaimCompiler)
3. Verify claims (VerificationRouter)
4. Aggregate into TypedVerdicts
5. Classify failure severity

In [15]:
# Generate questions and set up pipeline
doe_questions = bench.generate_questions()
compiler = ClaimCompiler(store=store)

# Full pipeline on all 640 questions
all_verdicts = []
all_failures = []

for i, dq in enumerate(doe_questions):
    q = dq.question
    agent_answer = str(q.ground_truth)  # correct answer

    # Step 1: Compile claims
    claims = compiler.compile(
        llm_answer=agent_answer,
        question_metadata=q.metadata,
        question_category=q.category,
        run_id="ch10-full",
        question_id=f"q-{i:03d}",
        ground_truth=str(q.ground_truth),
        question_text=q.natural_language,
    )

    # Step 2: Verify claims
    sub_verdicts = router.route_all(claims)

    # Step 3: Aggregate into TypedVerdict
    verdict = TypedVerdict.from_sub_verdicts(
        claims=claims,
        sub_verdicts=sub_verdicts,
        exact_correct=True,
        exact_detail="ground truth",
        run_id="ch10-full",
        question_id=f"q-{i:03d}",
    )
    all_verdicts.append(verdict)
    all_failures.extend(verdict.failures)

total_claims = sum(v.n_claims for v in all_verdicts)
total_passed = sum(v.n_passed for v in all_verdicts)
total_failed = sum(v.n_failed for v in all_verdicts)

print("Full pipeline results (640 questions):")
print(f"  TypedVerdicts:  {len(all_verdicts)}")
print(f"  Total claims:   {total_claims}")
print(f"  Claims passed:  {total_passed} ({total_passed/total_claims*100:.1f}%)")
print(f"  Claims failed:  {total_failed} ({total_failed/total_claims*100:.1f}%)")
print(f"  Total failures: {len(all_failures)}")

  Base questions: 160 (20 generators, max 10/cat)

  DOE expansion: 160 base x 4 runs = 640 questions
  Factors: ['clarity', 'entity_aliasing', 'reasoning_cue']
  [BackgroundTrain] Starting: 1 entities, 0 relations, 25 triples


  [BackgroundTrain] Done: Trained: 1 entities, 0 relations, 50 steps in 4.2s
  Store saved to doe_tracing_store/_doe_tracing
  [BackgroundTrain] Store saved


Full pipeline results (640 questions):
  TypedVerdicts:  640
  Total claims:   1156
  Claims passed:  456 (39.4%)
  Claims failed:  700 (60.6%)
  Total failures: 700


In [16]:
# Classify all failures
assessments_std = classifier.classify_all(all_failures, risk_tier="standard")
assessments_act = classifier.classify_all(all_failures, risk_tier="action_taking")

counts_std = classifier.count_by_level(assessments_std)
counts_act = classifier.count_by_level(assessments_act)

print(f"\nSeverity distribution ({len(all_failures)} failures):\n")
print(f"{'Level':<12s} {'Standard':>10s} {'Action-Taking':>15s}")
print("-" * 40)
for level in [SeverityLevel.CRITICAL, SeverityLevel.HIGH,
              SeverityLevel.MODERATE, SeverityLevel.LOW]:
    print(f"{level.value:<12s} {counts_std[level]:>10d} {counts_act[level]:>15d}")


Severity distribution (700 failures):

Level          Standard   Action-Taking
----------------------------------------
critical              0             640
high                640              40
moderate             40               0
low                  20              20


In [17]:
# Failure code breakdown with severity
code_counts = Counter(f.code.value for f in all_failures)

print("\nFailure codes with default severity:\n")
print(f"{'Code':<30s} {'Count':>6s} {'Base Severity':<14s}")
print("-" * 54)
for code_val, count in code_counts.most_common():
    code_enum = FailureCode(code_val)
    base_sev = CODE_SEVERITY_MAP.get(code_enum, SeverityLevel.LOW)
    print(f"{code_val:<30s} {count:>6d} {base_sev.value:<14s}")


Failure codes with default severity:

Code                            Count Base Severity 
------------------------------------------------------
fact_incorrect                    640 high          
holonomy_defect                    40 moderate      
verification_error                 20 low           


In [18]:
# Overall confidence distribution across TypedVerdicts
overall_confs = [v.overall_confidence for v in all_verdicts]
geo_confs = [v.geodesic_confidence for v in all_verdicts if v.geodesic_confidence > 0]

print("TypedVerdict confidence statistics (640 questions):\n")
print("Overall confidence:")
print(f"  Mean:   {np.mean(overall_confs):.4f}")
print(f"  Median: {np.median(overall_confs):.4f}")
print(f"  Min:    {np.min(overall_confs):.4f}")
print(f"  Max:    {np.max(overall_confs):.4f}")

if geo_confs:
    print("\nGeodesic confidence (questions with graph claims):")
    print(f"  Count:  {len(geo_confs)}")
    print(f"  Mean:   {np.mean(geo_confs):.4f}")
    print(f"  Median: {np.median(geo_confs):.4f}")

# Questions with zero claims
zero_claim = sum(1 for v in all_verdicts if v.n_claims == 0)
print(f"\nQuestions with 0 claims: {zero_claim}/{len(all_verdicts)}")

TypedVerdict confidence statistics (640 questions):

Overall confidence:
  Mean:   0.7129
  Median: 0.7000
  Min:    0.5642
  Max:    0.8008

Geodesic confidence (questions with graph claims):
  Count:  320
  Mean:   0.2662
  Median: 0.2675

Questions with 0 claims: 204/640


In [19]:
# Sample TypedVerdict with failures — show full audit trail
failed_verdicts = [v for v in all_verdicts if v.n_failed > 0]
if failed_verdicts:
    sample_v = failed_verdicts[0]
    print("=== Sample TypedVerdict (with failures) ===")
    print(f"  question_id:        {sample_v.question_id}")
    print(f"  overall_confidence: {sample_v.overall_confidence:.4f}")
    print(f"  geodesic_conf:      {sample_v.geodesic_confidence:.4f}")
    print(f"  n_claims:           {sample_v.n_claims}")
    print(f"  n_passed:           {sample_v.n_passed}")
    print(f"  n_failed:           {sample_v.n_failed}")
    print(f"  failure_codes:      {sample_v.failure_codes}")
    print("\n  Verifier summary:")
    for vname, counts in sample_v.verifier_summary.items():
        print(f"    {vname}: pass={counts['passed']} fail={counts['failed']}")
    print("\n  Severity classifications (action_taking tier):")
    for f in sample_v.failures[:3]:
        a = classifier.classify(f, risk_tier="action_taking")
        print(f"    {f.code.value}: {a.level.value} "
              f"(impact={a.impact_score:.2f}, "
              f"detect={a.detectability_score:.2f})")

=== Sample TypedVerdict (with failures) ===
  question_id:        q-000
  overall_confidence: 0.7895
  geodesic_conf:      0.2984
  n_claims:           9
  n_passed:           5
  n_failed:           4
  failure_codes:      ['fact_incorrect']

  Verifier summary:
    GraphVerifier: pass=5 fail=4

  Severity classifications (action_taking tier):
    fact_incorrect: critical (impact=1.00, detect=0.60)
    fact_incorrect: critical (impact=1.00, detect=0.60)
    fact_incorrect: critical (impact=1.00, detect=0.60)


---

## Exercises

### Exercise 10.1

Construct a custom `SeverityClassifier` that promotes `EXACT_MISMATCH` from LOW to HIGH (for a regulated environment where exact numeric accuracy is critical). Re-classify the 640-question failures and compare the severity distribution to the default classifier.

### Exercise 10.2

Build a `TypedVerdict` from a multi-hop question that produces both FactClaims and a RuleClaim. Inspect the `verifier_summary` property. Which verifier has the lowest pass rate? What does this tell you about the agent's multi-hop reasoning?

### Exercise 10.3

The composite formula weights impact at 0.50, recoverability at 0.25 and detectability at 0.25. Argue for an alternative weighting that would be more appropriate for a safety-critical medical AI agent. Implement it by passing custom weights to the composite calculation and show how the severity distribution changes.

In [20]:
# === Exercise 10.1: Your code here ===
# Hint:
# custom = SeverityClassifier(
#     store=store,
#     custom_map={FailureCode.EXACT_MISMATCH: SeverityLevel.HIGH},
# )
# Compare custom.count_by_level(...) vs default

In [21]:
# === Exercise 10.2: Your code here ===
# Hint: find a multi_hop question, compile + verify + TypedVerdict

In [22]:
# === Exercise 10.3: Your code here ===
# Hint: the composite formula is in SeverityClassifier.classify()

---

## Summary

This chapter completed Part III — the typed verification pipeline — by introducing three components that transform raw verification results into governance-ready outputs:

1. **FailureCode taxonomy** — 20+ codes in 7 categories (semantic, graph, logic, numeric, artifact, policy, process), each mapping to a specific GMS verification route.
2. **TypedVerdict** — aggregates per-claim sub-verdicts into an overall verdict with backwards-compatible geometric scores (`geodesic_confidence`, `tension_score`, `holonomy_score`, `overall_confidence`) and a full audit trail (claims, sub_verdicts, failures).
3. **SeverityClassifier** — maps each failure to CRITICAL / HIGH / MODERATE / LOW using failure code, risk tier and GMS geometric signals. The `action_taking` tier escalates moderate and high failures by one level.
4. **SeverityAssessment** — three-dimensional composite (impact x recoverability x detectability) provides nuanced severity signals. Geometric modulation escalates when composite >= 0.85.
5. The complete pipeline (claims -> verification -> verdict -> severity) produces an **audit packet** suitable for model risk management, regulatory review and release gating.

**Next:** Part IV introduces black-box agentic testing — SUT adapters (Chapter 11), tool gateways with fault injection (Chapter 12), policy enforcement (Chapter 13), environment simulation (Chapter 14) and composite oracles (Chapter 15).